In [ ]:
import pandas as pd
import json
with open("../data/Bronze/licitaciones.json", "r", encoding="utf-8") as f:
    data = json.load(f)
with open("../data/Bronze/adjudicatarios_licitaciones.json", "r", encoding="utf-8") as f:
    data_2 = json.load(f)

In [52]:
# Revisión inicial de estructura
print(len(data))
print(len(data_2))


72018
71834


In [53]:
df_lic = pd.DataFrame(data)
df_adj = pd.DataFrame(data_2)

df_lic.shape, df_adj.shape

((72018, 44), (71834, 15))

In [54]:
df_lic.columns

Index(['licitacion_id', 'title', 'detail_url', 'updated', 'expediente',
       'pais_codigo', 'pais_nombre', 'tipo_contrato_codigo',
       'subtipo_contrato_codigo', 'lugar_ejecucion', 'lugar_ejecucion_codigo',
       'cpv_codes', 'organo_contratacion', 'estado_codigo',
       'fecha_publicacion', 'procedimiento_codigo',
       'sistema_contratacion_codigo', 'importe_estimado',
       'importe_sin_impuestos', 'importe_total', 'forma_presentacion_codigo',
       'financiacion_ue_codigo', 'financiacion_ue_nombre',
       'fuente_publicacion', 'tipo_tramitacion_codigo', 'presentacion_desde',
       'presentacion_hasta', 'presentacion_hora', 'notice_types',
       'organo_dir3', 'organo_nif', 'org_hierarchy', 'contrato_duracion',
       'contrato_duracion_unidad', 'ofertas_recibidas', 'pymes_ofertas',
       'pyme_adjudicada', 'adjudicatario', 'adjudicatario_nif',
       'fecha_scraping', 'scraping_fuente_url', 'scraping_nota',
       'scraping_fuente_tipo', 'scraping_observaciones'],
   

In [55]:
df_adj.columns

Index(['licitacion_id', 'expediente', 'titulo', 'estado', 'fecha_publicacion',
       'tipo_contrato', 'organo_contratacion', 'importe_total',
       'importe_sin_impuestos', 'importe_estimado', 'adjudicatario',
       'adjudicatario_nif', 'url', 'fuente_scraping',
       'observaciones_scraping'],
      dtype='object')

In [56]:
# Cantidad de IDs únicos en cada base
df_lic["licitacion_id"].nunique(), df_adj["licitacion_id"].nunique()

(72018, 71834)

In [57]:
cpv_los_tilos = [
    "85100000",  # servicios de salud
    "85120000",  # servicios médicos
    "85121000",
    "85121100",  # medicina general
    "85121200",  # médicos especialistas
    "85121210",  # ginecología
    "85121231",  # cardiología
    "85121232",  # neumología
    "85121240",  # otorrino
    "85121270",  # psiquiatría/psicología
    "85121281",  # oftalmología
    "85121282",  # dermatología
    "85121283",  # ortopedia/traumatología
    "85121292",  # urología
    "85140000",  # servicios varios de salud
    "85141200",  # enfermería
    "85142100",  # fisioterapia
    "85145000",  # laboratorios médicos
    "85147000",  # sanidad empresarial / reconocimientos
    "85148000",  # análisis médicos
    "85150000",  # diagnóstico por imagen
]

In [ ]:
df_lic_tilos = df_lic[
    df_lic["cpv_codes"].fillna("").apply(
    lambda x: any(
        cpv in str(x)
        for cpv in cpv_los_tilos
    )
)
    
].copy()

In [59]:
df_lic_tilos.shape

(12621, 44)

In [60]:
df_lic_tilos[["licitacion_id", "title", "cpv_codes", "importe_total"]].head()

,licitacion_id,title,cpv_codes,importe_total
27,9239cad837632514,Servicio de prevención de adicciones en ocio y...,85140000,
45,448f6964aebcd861,realización de pruebas analíticas por laborato...,85145000,
80,f37ce812644369e9,Unidad móvil de prevención y diagnóstico preco...,85140000,
142,58a2b0f21be4191c,Servicios de asistencia psiquiátrica a persona...,85121270,
190,6ca4112ec156b9d7,Servicio de realización de diagnósticos citoló...,85145000,


In [61]:
ids_los_tilos = df_lic_tilos["licitacion_id"].unique()

df_adj_tilos = df_adj[
    df_adj["licitacion_id"].isin(ids_los_tilos)
].copy()

In [62]:
df_lic.shape, df_lic_tilos.shape, df_adj.shape, df_adj_tilos.shape

((72018, 44), (12621, 44), (71834, 15), (12619, 15))

In [63]:
ids_lic_tilos = set(df_lic_tilos["licitacion_id"])
ids_adj_tilos = set(df_adj_tilos["licitacion_id"])

ids_faltan_en_adj = ids_lic_tilos - ids_adj_tilos

df_lic_tilos = df_lic_tilos[
    ~df_lic_tilos["licitacion_id"].isin(ids_faltan_en_adj)
].copy()

In [64]:
df_lic_tilos.shape, df_adj_tilos.shape

((12619, 44), (12619, 15))

Se eliminaron dos licitaciones filtradas por CPV Los Tilos que no presentaban correspondencia en la tabla de adjudicatarios. Dado que representan una proporción marginal del conjunto filtrado y que el análisis posterior requiere consistencia entre ambas fuentes, se decidió excluirlas del universo final de análisis.

In [65]:
na_lic = (
    df_lic_tilos
    .isna()
    .sum()
    .reset_index()
)

na_lic.columns = ["columna", "n_na"]
na_lic["pct_na"] = (na_lic["n_na"] / len(df_lic_tilos) * 100).round(2)

na_lic = na_lic[na_lic["n_na"] > 0].sort_values("n_na", ascending=False)

na_lic

,columna,n_na,pct_na
43,scraping_observaciones,12619,100.00
39,fecha_scraping,12619,100.00
42,scraping_fuente_tipo,12619,100.00
40,scraping_fuente_url,12609,99.92
41,scraping_nota,12609,99.92
3,updated,10,0.08
5,pais_codigo,10,0.08
6,pais_nombre,10,0.08
8,subtipo_contrato_codigo,10,0.08
10,lugar_ejecucion_codigo,10,0.08


In [66]:
na_adj = (
    df_adj_tilos
    .isna()
    .sum()
    .reset_index()
)

na_adj.columns = ["columna", "n_na"]
na_adj["pct_na"] = (na_adj["n_na"] / len(df_adj_tilos) * 100).round(2)

na_adj = na_adj[na_adj["n_na"] > 0].sort_values("n_na", ascending=False)

na_adj

,columna,n_na,pct_na


In [67]:
umbral_na = 90

cols_eliminar_na = na_lic.loc[
    na_lic["pct_na"] > umbral_na,
    "columna"
].tolist()

cols_eliminar_na

['scraping_observaciones',
 'fecha_scraping',
 'scraping_fuente_tipo',
 'scraping_fuente_url',
 'scraping_nota']

In [68]:
df_lic_tilos = df_lic_tilos.drop(columns=cols_eliminar_na)

In [69]:
na_lic_post = (
    df_lic_tilos
    .isna()
    .sum()
    .reset_index()
)

na_lic_post.columns = ["columna", "n_na"]
na_lic_post["pct_na"] = (na_lic_post["n_na"] / len(df_lic_tilos) * 100).round(2)

na_lic_post = na_lic_post[na_lic_post["n_na"] > 0].sort_values("n_na", ascending=False)

na_lic_post

,columna,n_na,pct_na
3,updated,10,0.08
5,pais_codigo,10,0.08
6,pais_nombre,10,0.08
8,subtipo_contrato_codigo,10,0.08
9,lugar_ejecucion,10,0.08
10,lugar_ejecucion_codigo,10,0.08
12,organo_contratacion,10,0.08
14,fecha_publicacion,10,0.08
15,procedimiento_codigo,10,0.08
16,sistema_contratacion_codigo,10,0.08


# Revisar columnas

In [70]:
df_lic_tilos = df_lic_tilos.rename(columns={"title": "titulo"})

In [71]:
# Columnas de cada DataFrame
cols_lic = set(df_lic_tilos.columns)
cols_adj = set(df_adj_tilos.columns)

# Columnas comunes
cols_comunes = sorted(cols_lic & cols_adj)

# Columnas que solo están en df_lic_tilos
cols_solo_lic = sorted(cols_lic - cols_adj)

# Columnas que solo están en df_adj_tilos
cols_solo_adj = sorted(cols_adj - cols_lic)

print("Columnas comunes:")
print(cols_comunes)

print("\nColumnas solo en df_lic_tilos:")
print(cols_solo_lic)

print("\nColumnas solo en df_adj_tilos:")
print(cols_solo_adj)

Columnas comunes:
['adjudicatario', 'adjudicatario_nif', 'expediente', 'fecha_publicacion', 'importe_estimado', 'importe_sin_impuestos', 'importe_total', 'licitacion_id', 'organo_contratacion', 'titulo']

Columnas solo en df_lic_tilos:
['contrato_duracion', 'contrato_duracion_unidad', 'cpv_codes', 'detail_url', 'estado_codigo', 'financiacion_ue_codigo', 'financiacion_ue_nombre', 'forma_presentacion_codigo', 'fuente_publicacion', 'lugar_ejecucion', 'lugar_ejecucion_codigo', 'notice_types', 'ofertas_recibidas', 'org_hierarchy', 'organo_dir3', 'organo_nif', 'pais_codigo', 'pais_nombre', 'presentacion_desde', 'presentacion_hasta', 'presentacion_hora', 'procedimiento_codigo', 'pyme_adjudicada', 'pymes_ofertas', 'sistema_contratacion_codigo', 'subtipo_contrato_codigo', 'tipo_contrato_codigo', 'tipo_tramitacion_codigo', 'updated']

Columnas solo en df_adj_tilos:
['estado', 'fuente_scraping', 'observaciones_scraping', 'tipo_contrato', 'url']


In [72]:
resumen_columnas = pd.DataFrame({
    "tipo": (
        ["comun"] * len(cols_comunes) +
        ["solo_df_lic_tilos"] * len(cols_solo_lic) +
        ["solo_df_adj_tilos"] * len(cols_solo_adj)
    ),
    "columna": cols_comunes + cols_solo_lic + cols_solo_adj
})

resumen_columnas

,tipo,columna
0,comun,adjudicatario
1,comun,adjudicatario_nif
2,comun,expediente
3,comun,fecha_publicacion
4,comun,importe_estimado
5,comun,importe_sin_impuestos
6,comun,importe_total
7,comun,licitacion_id
8,comun,organo_contratacion
9,comun,titulo


In [73]:
# Columnas comunes entre ambas tablas
cols_comunes = sorted(set(df_lic_tilos.columns) & set(df_adj_tilos.columns))

# Resumen de NA en columnas comunes
na_cols_comunes = pd.DataFrame({
    "columna": cols_comunes,
    "na_df_lic_tilos": [df_lic_tilos[col].isna().sum() for col in cols_comunes],
    "pct_na_df_lic_tilos": [
        round(df_lic_tilos[col].isna().mean() * 100, 2) for col in cols_comunes
    ],
    "na_df_adj_tilos": [df_adj_tilos[col].isna().sum() for col in cols_comunes],
    "pct_na_df_adj_tilos": [
        round(df_adj_tilos[col].isna().mean() * 100, 2) for col in cols_comunes
    ],
})

na_cols_comunes.sort_values(
    by=["na_df_lic_tilos", "na_df_adj_tilos"],
    ascending=False
)

,columna,na_df_lic_tilos,pct_na_df_lic_tilos,na_df_adj_tilos,pct_na_df_adj_tilos
3,fecha_publicacion,10,0.08,0,0.0
8,organo_contratacion,10,0.08,0,0.0
4,importe_estimado,5,0.04,0,0.0
5,importe_sin_impuestos,5,0.04,0,0.0
6,importe_total,4,0.03,0,0.0
0,adjudicatario,0,0.00,0,0.0
1,adjudicatario_nif,0,0.00,0,0.0
2,expediente,0,0.00,0,0.0
7,licitacion_id,0,0.00,0,0.0
9,titulo,0,0.00,0,0.0


In [74]:
# Columnas comunes, quitando la llave
cols_comunes = sorted(
    (set(df_lic_tilos.columns) & set(df_adj_tilos.columns)) - {"licitacion_id"}
)

# Pasamos df_adj_tilos a índice por licitacion_id para alinear correctamente
df_adj_index = df_adj_tilos.set_index("licitacion_id")

# Llenar NA de df_lic_tilos con datos de df_adj_tilos
for col in cols_comunes:
    df_lic_tilos[col] = df_lic_tilos[col].fillna(
        df_lic_tilos["licitacion_id"].map(df_adj_index[col])
    )

In [75]:
na_cols_comunes_post = pd.DataFrame({
    "columna": cols_comunes,
    "na_df_lic_tilos": [df_lic_tilos[col].isna().sum() for col in cols_comunes],
    "pct_na_df_lic_tilos": [
        round(df_lic_tilos[col].isna().mean() * 100, 2) for col in cols_comunes
    ],
    "na_df_adj_tilos": [df_adj_tilos[col].isna().sum() for col in cols_comunes],
    "pct_na_df_adj_tilos": [
        round(df_adj_tilos[col].isna().mean() * 100, 2) for col in cols_comunes
    ],
})

na_cols_comunes_post.sort_values(
    by=["na_df_lic_tilos", "na_df_adj_tilos"],
    ascending=False
)

,columna,na_df_lic_tilos,pct_na_df_lic_tilos,na_df_adj_tilos,pct_na_df_adj_tilos
0,adjudicatario,0,0.0,0,0.0
1,adjudicatario_nif,0,0.0,0,0.0
2,expediente,0,0.0,0,0.0
3,fecha_publicacion,0,0.0,0,0.0
4,importe_estimado,0,0.0,0,0.0
5,importe_sin_impuestos,0,0.0,0,0.0
6,importe_total,0,0.0,0,0.0
7,organo_contratacion,0,0.0,0,0.0
8,titulo,0,0.0,0,0.0


In [76]:
# Columnas de df_adj_tilos que NO están en df_lic_tilos
cols_solo_adj = sorted(set(df_adj_tilos.columns) - set(df_lic_tilos.columns))

cols_solo_adj

['estado', 'fuente_scraping', 'observaciones_scraping', 'tipo_contrato', 'url']

In [77]:
df_adj_join = df_adj_tilos[["licitacion_id"] + cols_solo_adj].copy()

df_tilos = df_lic_tilos.merge(
    df_adj_join,
    on="licitacion_id",
    how="left"
)

In [78]:
df_tilos.columns[df_tilos.columns.duplicated()]

Index([], dtype='object')

# Validación data

In [79]:
df_tilos.shape

(12619, 44)

In [80]:
df_tilos.columns

Index(['licitacion_id', 'titulo', 'detail_url', 'updated', 'expediente',
       'pais_codigo', 'pais_nombre', 'tipo_contrato_codigo',
       'subtipo_contrato_codigo', 'lugar_ejecucion', 'lugar_ejecucion_codigo',
       'cpv_codes', 'organo_contratacion', 'estado_codigo',
       'fecha_publicacion', 'procedimiento_codigo',
       'sistema_contratacion_codigo', 'importe_estimado',
       'importe_sin_impuestos', 'importe_total', 'forma_presentacion_codigo',
       'financiacion_ue_codigo', 'financiacion_ue_nombre',
       'fuente_publicacion', 'tipo_tramitacion_codigo', 'presentacion_desde',
       'presentacion_hasta', 'presentacion_hora', 'notice_types',
       'organo_dir3', 'organo_nif', 'org_hierarchy', 'contrato_duracion',
       'contrato_duracion_unidad', 'ofertas_recibidas', 'pymes_ofertas',
       'pyme_adjudicada', 'adjudicatario', 'adjudicatario_nif', 'estado',
       'fuente_scraping', 'observaciones_scraping', 'tipo_contrato', 'url'],
      dtype='object')

In [81]:
[col for col in df_tilos.columns if col.endswith("_x") or col.endswith("_y")]

[]

In [82]:
df_tilos["licitacion_id"].isna().sum()

np.int64(0)

In [83]:
valores_vacios = ["", " ", "None", "none", "NULL", "null", "NaN", "nan"]

resumen_vacios = []

for col in df_tilos.columns:
    serie = df_tilos[col]
    
    # NA reales
    n_na = serie.isna().sum()
    
    # Vacíos tipo texto, pero SOLO en valores que no son NA reales
    if serie.dtype == "object":
        serie_no_na = serie[serie.notna()]
        serie_str = serie_no_na.astype(str).str.strip()
        n_vacios_texto = serie_str.isin(valores_vacios).sum()
    else:
        n_vacios_texto = 0
    
    n_total_faltantes = n_na + n_vacios_texto
    pct_total_faltantes = round(n_total_faltantes / len(df_tilos) * 100, 2)
    
    resumen_vacios.append({
        "columna": col,
        "dtype": serie.dtype,
        "n_na": n_na,
        "n_vacios_texto": n_vacios_texto,
        "n_total_faltantes": n_total_faltantes,
        "pct_total_faltantes": pct_total_faltantes
    })

resumen_vacios_df = pd.DataFrame(resumen_vacios)

resumen_vacios_df = (
    resumen_vacios_df
    .query("n_total_faltantes > 0")
    .sort_values("n_total_faltantes", ascending=False)
)

resumen_vacios_df

,columna,dtype,n_na,n_vacios_texto,n_total_faltantes,pct_total_faltantes
41,observaciones_scraping,object,0,12619,12619,100.00
25,presentacion_desde,object,10,12609,12619,100.00
35,pymes_ofertas,object,10,12608,12618,99.99
36,pyme_adjudicada,object,10,12608,12618,99.99
6,pais_nombre,object,10,12606,12616,99.98
5,pais_codigo,object,10,12606,12616,99.98
8,subtipo_contrato_codigo,object,10,12606,12616,99.98
9,lugar_ejecucion,object,10,12606,12616,99.98
30,organo_nif,object,10,12606,12616,99.98
19,importe_total,object,0,12610,12610,99.93


In [84]:
# Columnas con más del 90% de faltantes
cols_eliminar_90 = resumen_vacios_df.loc[
    resumen_vacios_df["pct_total_faltantes"] > 90,
    "columna"
].tolist()

cols_eliminar_90

['observaciones_scraping',
 'presentacion_desde',
 'pymes_ofertas',
 'pyme_adjudicada',
 'pais_nombre',
 'pais_codigo',
 'subtipo_contrato_codigo',
 'lugar_ejecucion',
 'organo_nif',
 'importe_total',
 'fuente_scraping',
 'financiacion_ue_codigo',
 'financiacion_ue_nombre',
 'tipo_tramitacion_codigo',
 'sistema_contratacion_codigo',
 'forma_presentacion_codigo',
 'org_hierarchy',
 'importe_estimado']

In [85]:
df_tilos = df_tilos.drop(columns=cols_eliminar_90)

In [86]:
df_tilos.shape

(12619, 26)

Se realizó una validación ampliada de valores faltantes, considerando no solo valores nulos estándar (NaN), sino también campos vacíos provenientes del proceso de scraping, tales como cadenas vacías, espacios en blanco y textos equivalentes a valores nulos ("nan", "None" o "null"). A partir de esta revisión, se identificaron variables con más del 90% de información ausente dentro del subconjunto filtrado para Clínica Los Tilos.

Dado que estas variables presentan una disponibilidad de datos insuficiente para aportar valor analítico y podrían introducir ruido, sesgos o interpretaciones poco fiables en etapas posteriores, se decidió eliminarlas del dataset consolidado. Esta decisión permite conservar únicamente variables con un nivel mínimo de completitud y mejora la calidad del conjunto de datos utilizado para el análisis exploratorio, temporal, geográfico y competitivo

In [87]:
import pandas as pd
dic_cpv = pd.read_excel(
    "../data/CPV_Tabla_descripcion.xlsx",
    sheet_name="Hoja1"
)



In [88]:
dic_cpv["cpv_codigo_limpio"] = (
    dic_cpv["Código CPV"]
    .str.split("-")
    .str[0]
)

In [89]:
dic_cpv.head()

,Código CPV,Código 8 dígitos,Dígito control,División (2 dígitos),Grupo (3 dígitos),Clase (4 dígitos),Categoría (5 dígitos),Nivel,Descripción,Página PDF,cpv_codigo_limpio
0,03000000-1,3000000,1,3,30,300,3000,División,"Productos de la agricultura, ganadería, pesca,...",2,03000000
1,03100000-2,3100000,2,3,31,310,3100,Grupo,Productos de la agricultura y horticultura,2,03100000
2,03110000-5,3110000,5,3,31,311,3110,Clase,"Cultivos, productos comerciales de jardinería ...",2,03110000
3,03111000-2,3111000,2,3,31,311,3111,Categoría,Semillas,2,03111000
4,03111100-3,3111100,3,3,31,311,3111,Detalle,Soja,2,03111100


In [90]:
df_tilos["cpv_codes"] = (
    df_tilos["cpv_codes"]
    .astype(str)
    .str.strip()
    .str.zfill(8)
)

dic_cpv["cpv_codigo_limpio"] = (
    dic_cpv["cpv_codigo_limpio"]
    .astype(str)
    .str.strip()
    .str.zfill(8)
)

In [91]:
dic_cpv_join = dic_cpv[
    [
        "cpv_codigo_limpio",
        "Nivel",
        "Descripción"
    ]
].copy()

In [92]:
df_tilos = df_tilos.merge(
    dic_cpv_join,
    left_on="cpv_codes",
    right_on="cpv_codigo_limpio",
    how="left"
)

In [93]:
df_tilos["Descripción"].isna().sum()

np.int64(136)

In [94]:
cpv_sin_match = (
    df_tilos.loc[df_tilos["Descripción"].isna(), "cpv_codes"]
    .value_counts()
    .reset_index()
)

cpv_sin_match.columns = ["cpv_codes", "n"]
cpv_sin_match

,cpv_codes,n
0,85121200;85100000,11
1,85141200;85141000,8
2,85100000;71317000,5
3,85100000;85140000,5
4,85148000;85147000;85140000;71317200;71317210,5
...,...,...
78,71317000;85140000;85147000;85148000;71317210,1
79,71317200;80562000;85100000;80560000;85140000;7...,1
80,73210000;71900000;60100000;85111820;64120000;7...,1
81,33100000;85121200,1


In [95]:
# Diccionario código CPV -> descripción
map_cpv_desc = dict(
    zip(dic_cpv["cpv_codigo_limpio"], dic_cpv["Descripción"])
)

map_cpv_nivel = dict(
    zip(dic_cpv["cpv_codigo_limpio"], dic_cpv["Nivel"])
)

In [96]:
def obtener_descripciones_cpv(cpv_string):
    if pd.isna(cpv_string):
        return None
    
    codigos = [
        str(c).strip().zfill(8)
        for c in str(cpv_string).split(";")
        if str(c).strip() != ""
    ]
    
    descripciones = [
        map_cpv_desc.get(c)
        for c in codigos
        if map_cpv_desc.get(c) is not None
    ]
    
    if len(descripciones) == 0:
        return None
    
    return "; ".join(sorted(set(descripciones)))


def obtener_niveles_cpv(cpv_string):
    if pd.isna(cpv_string):
        return None
    
    codigos = [
        str(c).strip().zfill(8)
        for c in str(cpv_string).split(";")
        if str(c).strip() != ""
    ]
    
    niveles = [
        map_cpv_nivel.get(c)
        for c in codigos
        if map_cpv_nivel.get(c) is not None
    ]
    
    if len(niveles) == 0:
        return None
    
    return "; ".join(sorted(set(niveles)))

In [97]:
df_tilos["cpv_descripcion"] = df_tilos["cpv_codes"].apply(obtener_descripciones_cpv)
df_tilos["cpv_nivel"] = df_tilos["cpv_codes"].apply(obtener_niveles_cpv)

In [98]:
cols_borrar = ["Descripción", "Nivel", "cpv_codigo_limpio"]

df_tilos = df_tilos.drop(
    columns=[col for col in cols_borrar if col in df_tilos.columns]
)

In [99]:
df_tilos["cpv_descripcion"].isna().sum()

np.int64(0)

In [100]:
df_tilos.shape

(12619, 28)

In [101]:
cpv_los_tilos_set = set(cpv_los_tilos)

def filtrar_cpv_los_tilos(cpv_string):
    if pd.isna(cpv_string):
        return None
    
    codigos = [
        str(c).strip().zfill(8)
        for c in str(cpv_string).split(";")
        if str(c).strip() != ""
    ]
    
    codigos_filtrados = [
        c for c in codigos
        if c in cpv_los_tilos_set
    ]
    
    if len(codigos_filtrados) == 0:
        return None
    
    return ";".join(sorted(set(codigos_filtrados)))

In [102]:
df_tilos["cpv_codes"] = df_tilos["cpv_codes"].apply(filtrar_cpv_los_tilos)

In [103]:
df_tilos["cpv_codes"].isna().sum()

np.int64(0)

In [104]:
map_cpv_desc = dict(
    zip(dic_cpv["cpv_codigo_limpio"], dic_cpv["Descripción"])
)

map_cpv_nivel = dict(
    zip(dic_cpv["cpv_codigo_limpio"], dic_cpv["Nivel"])
)

def obtener_descripciones_cpv(cpv_string):
    if pd.isna(cpv_string):
        return None
    
    codigos = [
        str(c).strip().zfill(8)
        for c in str(cpv_string).split(";")
        if str(c).strip() != ""
    ]
    
    descripciones = [
        map_cpv_desc.get(c)
        for c in codigos
        if map_cpv_desc.get(c) is not None
    ]
    
    if len(descripciones) == 0:
        return None
    
    return "; ".join(sorted(set(descripciones)))


def obtener_niveles_cpv(cpv_string):
    if pd.isna(cpv_string):
        return None
    
    codigos = [
        str(c).strip().zfill(8)
        for c in str(cpv_string).split(";")
        if str(c).strip() != ""
    ]
    
    niveles = [
        map_cpv_nivel.get(c)
        for c in codigos
        if map_cpv_nivel.get(c) is not None
    ]
    
    if len(niveles) == 0:
        return None
    
    return "; ".join(sorted(set(niveles)))

In [105]:
df_tilos["cpv_descripcion"] = df_tilos["cpv_codes"].apply(obtener_descripciones_cpv)
df_tilos["cpv_nivel"] = df_tilos["cpv_codes"].apply(obtener_niveles_cpv)

In [106]:
df_tilos[["cpv_codes", "cpv_descripcion", "cpv_nivel","tipo_contrato"]].head(20)

,cpv_codes,cpv_descripcion,cpv_nivel,tipo_contrato
0,85140000,Servicios varios de salud,Clase,Servicios
1,85145000,Servicios prestados por laboratorios médicos,Categoría,Servicios
2,85140000,Servicios varios de salud,Clase,Servicios
3,85121270,Servicios psiquiátricos o psicológicos,Detalle,Concesion de obras
4,85145000,Servicios prestados por laboratorios médicos,Categoría,Servicios
5,85140000,Servicios varios de salud,Clase,Servicios
6,85121200,Servicios de médicos especialistas,Detalle,Concesion de obras
7,85121200,Servicios de médicos especialistas,Detalle,Concesion de obras
8,85100000,Servicios de salud,Grupo,Servicios
9,85100000,Servicios de salud,Grupo,Servicios


In [107]:
df_tilos["tipo_contrato"].value_counts(dropna=False)

tipo_contrato
Obras                     9608
Servicios                 2938
Suministros                 38
7                           14
Concesion de obras          11
999                          7
Concesion de servicios       3
Name: count, dtype: int64

In [108]:
df_tilos["cpv_descripcion"].value_counts(dropna=False)

cpv_descripcion
Servicios de salud                                                                                                                                       12054
Servicios varios de salud                                                                                                                                   97
Servicios prestados por laboratorios médicos                                                                                                                84
Servicios de médicos especialistas                                                                                                                          55
Servicios psiquiátricos o psicológicos                                                                                                                      47
Servicios ginecológicos u obstétricos                                                                                                                       47
Servicios de análisis médicos 

In [109]:
otros = [
    "999",
    "7",
    "32",
    "40",
    "2"
]

df_tilos["tipo_contrato"] = (
    df_tilos["tipo_contrato"]
    .replace(otros, "Otros / No clasificado")
)

In [110]:
df_tilos["tipo_contrato"].value_counts(dropna=False)

tipo_contrato
Obras                     9608
Servicios                 2938
Suministros                 38
Otros / No clasificado      21
Concesion de obras          11
Concesion de servicios       3
Name: count, dtype: int64

# Tipo de columnas

In [111]:
df_tilos.dtypes

licitacion_id               object
titulo                      object
detail_url                  object
updated                     object
expediente                  object
tipo_contrato_codigo        object
lugar_ejecucion_codigo      object
cpv_codes                   object
organo_contratacion         object
estado_codigo               object
fecha_publicacion           object
procedimiento_codigo        object
importe_sin_impuestos       object
fuente_publicacion          object
presentacion_hasta          object
presentacion_hora           object
notice_types                object
organo_dir3                 object
contrato_duracion           object
contrato_duracion_unidad    object
ofertas_recibidas           object
adjudicatario               object
adjudicatario_nif           object
estado                      object
tipo_contrato               object
url                         object
cpv_descripcion             object
cpv_nivel                   object
dtype: object

In [112]:
df_tilos["fecha_publicacion"] = (
    df_tilos["fecha_publicacion"]
    .astype("string")
    .str.strip()
    .str[:10]
)

df_tilos["fecha_publicacion"] = pd.to_datetime(
    df_tilos["fecha_publicacion"],
    format="%Y-%m-%d",
    errors="raise"
)

In [113]:
df_tilos["fecha_publicacion"].value_counts(dropna=False)

fecha_publicacion
2023-02-09    1810
2023-02-16    1386
2023-03-10    1261
2023-02-20     973
2023-02-13     966
              ... 
2026-03-16       1
2018-03-13       1
2018-03-22       1
2018-05-21       1
2018-01-25       1
Name: count, Length: 603, dtype: int64

In [114]:
df_tilos["presentacion_hasta"] = (
    df_tilos["presentacion_hasta"]
    .astype("string")
    .str.strip()
    .str[:10]
)

df_tilos["presentacion_hasta"] = pd.to_datetime(
    df_tilos["presentacion_hasta"],
    format="%Y-%m-%d",
    errors="raise"
)

In [115]:
df_tilos["presentacion_hasta"].dtype

dtype('<M8[ns]')

In [116]:
df_tilos["presentacion_hasta"].isna().sum()

np.int64(286)

In [117]:
df_tilos["presentacion_hasta"].value_counts(dropna=False)

presentacion_hasta
2023-02-09    1810
2023-02-16    1385
2023-03-10    1260
2023-02-20     973
2023-02-13     966
              ... 
2025-04-07       1
2019-04-22       1
2019-03-22       1
2019-03-21       1
2018-01-30       1
Name: count, Length: 513, dtype: int64

In [118]:
df_tilos["updated"] = (
    df_tilos["updated"]
    .astype("string")
    .str.strip()
    .str[:10]
)

df_tilos["updated"] = pd.to_datetime(
    df_tilos["updated"],
    format="%Y-%m-%d",
    errors="raise"
)

In [119]:
df_tilos["updated"].dtype

dtype('<M8[ns]')

In [120]:
df_tilos["updated"].isna().sum()

np.int64(10)

In [121]:
df_tilos["updated"].value_counts(dropna=False)

updated
2023-02-09    1541
2023-02-16    1461
2023-03-10    1328
2023-02-20     973
2023-02-13     965
              ... 
2026-04-28       1
2020-07-14       1
2020-07-31       1
2020-07-17       1
2018-03-01       1
Name: count, Length: 593, dtype: int64

In [122]:
df_tilos["importe_sin_impuestos"].dtype

dtype('O')

In [123]:
df_tilos["importe_sin_impuestos"].head(20)

0      82643.32
1     1159639.8
2        184000
3        315235
4         75200
5         24000
6        888044
7        742801
8        750000
9      21586.71
10        53123
11       341622
12        41765
13       243740
14        48274
15       112881
16     176631.6
17     102555.7
18      82491.2
19       715400
Name: importe_sin_impuestos, dtype: object

In [124]:
df_tilos["importe_sin_impuestos"].describe()

count     12619
unique     8387
top       16800
freq        157
Name: importe_sin_impuestos, dtype: object

In [125]:
importe_original = df_tilos["importe_sin_impuestos"].copy()

In [126]:
importe_original.notna().sum()

np.int64(12619)

In [127]:
importe_original.astype("string").str.strip().value_counts(dropna=False).head(20)

importe_sin_impuestos
16800       157
16128        95
17280        78
13824        50
8640         27
12096        25
17454.52     21
212          21
500          21
9000         21
400          20
1000         20
847          19
331.81       19
14820        19
             18
726          18
18000        18
2221.26      17
2879.8       17
Name: count, dtype: Int64

In [128]:
importe_convertido = (
    df_tilos["importe_sin_impuestos"]
    .astype("string")
    .str.replace(",", ".", regex=False)
    .str.strip()
)

importe_convertido = pd.to_numeric(
    importe_convertido,
    errors="raise"
)

In [129]:
comparacion_importe = pd.DataFrame({
    "original": importe_original,
    "convertido": importe_convertido
})

comparacion_importe.head(20)

,original,convertido
0,82643.32,82643.32
1,1159639.8,1159639.8
2,184000,184000.0
3,315235,315235.0
4,75200,75200.0
5,24000,24000.0
6,888044,888044.0
7,742801,742801.0
8,750000,750000.0
9,21586.71,21586.71


In [130]:
perdidos_conversion = comparacion_importe[
    comparacion_importe["original"].notna() &
    comparacion_importe["convertido"].isna()
]

perdidos_conversion.shape

(18, 2)

In [131]:
perdidos_conversion["original"].value_counts(dropna=False)

original
    18
Name: count, dtype: int64

In [132]:
df_tilos.loc[
    perdidos_conversion.index,
    ["licitacion_id", "titulo", "importe_sin_impuestos"]
]

,licitacion_id,titulo,importe_sin_impuestos
39,fa7097415ad24e0a,SERVICIOS DE TOMOGRAFÍA POR EMISIÓN DE POSITRO...,
137,b11de4c7a04f9993,Vigilancia de la Salud.,
672,ba08313fb13df148,"Contratación del servicio de diseño, coordinac...",
675,3e8e05fcb8c1896c,Oncotype,
682,2a8c612bb3a08e71,Servicio de diagnóstico por la imagen mediante...,
12141,0c049d31ff03c779,Servicio de pruebas diagnósticas mediante PET-...,
12142,7c490703b59d1d36,Servicios de diagnóstico por la imagen mediant...,
12305,a42c08702ce83eb0,Servicios profesionales de lectura y emisión d...,
12319,b1a7b1002204d79a,Servicio de diagnóstico por la imagen mediante...,
12326,b9b235ec49642c55,Contrato de servicios para la realización de d...,


In [133]:
df_tilos["importe_sin_impuestos"] = importe_convertido

In [134]:
df_tilos["importe_sin_impuestos"].dtype

Float64Dtype()

In [135]:
df_tilos["importe_sin_impuestos"].isna().sum()

np.int64(18)

In [136]:
df_tilos["importe_sin_impuestos"].describe()

count           12601.0
mean      180266.994296
std      3449792.813161
min                 0.0
25%              464.63
50%              2000.0
75%             10400.0
max         231886643.5
Name: importe_sin_impuestos, dtype: Float64

In [137]:
(df_tilos["importe_sin_impuestos"] == 0.0).sum()

np.int64(12)

In [138]:
df_tilos.loc[
    df_tilos["importe_sin_impuestos"] == 0.0,
    ["licitacion_id", "titulo", "importe_sin_impuestos", "tipo_contrato", "estado"]
].head(20)

,licitacion_id,titulo,importe_sin_impuestos,tipo_contrato,estado
77,55a30cba04c1f588,S'informa de la contractació anual programada ...,0.0,Obras,Anuncio previo
78,6f0592d401ec3f0e,S'informa de la contractació anual programada ...,0.0,Obras,Anuncio previo
247,6eff2cfddfcce6b0,Segundo Concierto Social para la Prestación de...,0.0,Otros / No clasificado,Publicada
299,8f4d03bebf4e6b6d,servicio de interrupción voluntaria del embara...,0.0,Servicios,Adjudicada
12170,3ee4a9583b398773,Servei de subministrament de plataformes genòm...,0.0,Servicios,Resuelta
12171,f236b17713f02bcf,Servei de manteniment correctiu dels seqüencia...,0.0,Servicios,Adjudicada
12374,334611b7ccd4ca9e,Prestación del Servicio de Podología en la Res...,0.0,Servicios,Adjudicada
12395,506f278aca322984,Servicio de Podología en el Centro de Particip...,0.0,Servicios,Resuelta
12469,3c54c60e78e1bd1f,Acuerdo marco para la adquisición de aparatos ...,0.0,Obras,Resuelta
12530,ffc88df1efe43c09,Servicio de Podología en el Centro de Particip...,0.0,Servicios,Publicada


In [139]:
df_tilos = df_tilos[df_tilos["importe_sin_impuestos"] != 0.0].copy()

In [140]:
(df_tilos["importe_sin_impuestos"] == 0.0).sum()

np.int64(0)

In [141]:
df_tilos.shape

(12589, 28)

Se eliminaron los registros con importe_sin_impuestos igual a 0, dado que no representan un valor económico válido para el análisis de licitaciones. Al tratarse de un número marginal de observaciones, su exclusión no afecta de forma significativa el tamaño del conjunto de datos, pero evita distorsiones en los análisis estadísticos, especialmente en medidas de tendencia central, dispersión, distribución de importes y detección de valores atípicos.

In [142]:
df_tilos.dtypes

licitacion_id                       object
titulo                              object
detail_url                          object
updated                     datetime64[ns]
expediente                          object
tipo_contrato_codigo                object
lugar_ejecucion_codigo              object
cpv_codes                           object
organo_contratacion                 object
estado_codigo                       object
fecha_publicacion           datetime64[ns]
procedimiento_codigo                object
importe_sin_impuestos              Float64
fuente_publicacion                  object
presentacion_hasta          datetime64[ns]
presentacion_hora                   object
notice_types                        object
organo_dir3                         object
contrato_duracion                   object
contrato_duracion_unidad            object
ofertas_recibidas                   object
adjudicatario                       object
adjudicatario_nif                   object
estado     

In [143]:
df_tilos["contrato_duracion"] = pd.to_numeric(
    df_tilos["contrato_duracion"],
    errors="raise"
)

df_tilos["ofertas_recibidas"] = pd.to_numeric(
    df_tilos["ofertas_recibidas"],
    errors="raise"
)

In [144]:
df_tilos.isna().sum().sort_values(ascending=False)

ofertas_recibidas           440
presentacion_hasta          265
contrato_duracion           171
updated                       5
fuente_publicacion            5
contrato_duracion_unidad      5
notice_types                  5
organo_dir3                   5
presentacion_hora             5
fecha_publicacion             5
procedimiento_codigo          5
lugar_ejecucion_codigo        5
organo_contratacion           0
estado_codigo                 0
tipo_contrato_codigo          0
cpv_codes                     0
expediente                    0
licitacion_id                 0
titulo                        0
detail_url                    0
importe_sin_impuestos         0
adjudicatario                 0
adjudicatario_nif             0
estado                        0
tipo_contrato                 0
url                           0
cpv_descripcion               0
cpv_nivel                     0
dtype: int64

In [145]:
df_tilos["licitacion_id"].duplicated().sum()

np.int64(0)

In [146]:
df_tilos["cpv_codes"].isna().sum()

np.int64(0)

In [147]:
df_tilos["cpv_descripcion"].value_counts(dropna=False)

cpv_descripcion
Servicios de salud                                                                                                                                       12049
Servicios varios de salud                                                                                                                                   91
Servicios prestados por laboratorios médicos                                                                                                                81
Servicios de médicos especialistas                                                                                                                          55
Servicios psiquiátricos o psicológicos                                                                                                                      47
Servicios ginecológicos u obstétricos                                                                                                                       46
Servicios de análisis médicos 

In [148]:
df_tilos["tipo_contrato"].value_counts(dropna=False)

tipo_contrato
Obras                     9605
Servicios                 2912
Suministros                 38
Otros / No clasificado      20
Concesion de obras          11
Concesion de servicios       3
Name: count, dtype: int64

In [149]:
df_tilos[
    df_tilos["adjudicatario"]
    .astype("string")
    .str.upper()
    .str.contains("TILOS", na=False)
][["licitacion_id", "titulo", "adjudicatario", "adjudicatario_nif", "importe_sin_impuestos"]]

,licitacion_id,titulo,adjudicatario,adjudicatario_nif,importe_sin_impuestos
12609,f14496dfcc8d5128,Contratación del servicio de Diagnóstico por l...,Ver detalle de la adjudicación (multi-lote; in...,No informado,631666.9
12611,fd6b2f590c512031,Servicio de asistencia sanitaria especializada...,"TILOSALUD, S.A.",A40213217,16375.0
12614,1967eade2fccb129,Contratación del servicio de Asistencia Sanita...,Ver detalle de la adjudicación (multi-lote; in...,No informado,270639.6
12616,3fc7125e9f8a3d95,Realización de procedimientos diagnósticos (ul...,"TILOSALUD, S.A.",A40213217,172000.0
12618,ddec9f7e2e9944dc,Servicio de reconocimientos médicos personal a...,"TILOSALUD, S.A.; ASPY PREVENCION.S.L.U.",A40213217; B98844574,15975.0


In [150]:
n_los_tilos = (
    df_tilos["adjudicatario"]
    .astype("string")
    .str.upper()
    .str.contains("TILOS", na=False)
    .sum()
)

n_los_tilos

np.int64(5)

In [ ]:
#import pyarrow
#df_tilos.to_parquet(
#    "../data/df_tilos_limpio.parquet",
#    index=False
#)